# Régression des ventes Walmart — notebook nettoyé

Copie de travail indépendante de `04_walmart_sales_lr + cv + L1 L2.ipynb`. Elle reprend le même objectif et applique la consigne des 3 sigmas aux quatre variables numériques concernées. Elle conserve aussi les corrections d’encodage des catégories et de stabilisation de Lasso. Le notebook source n’a pas été modifié.

## 1. Reprendre les imports et la configuration existants

On garde pandas, numpy et scikit-learn, déjà utilisés dans le notebook source. Le notebook nécessite également matplotlib uniquement si l’on ajoute des graphiques ; aucun graphique n’est requis pour l’évaluation ci-dessous.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
TARGET = "weekly_sales"
DATA_PATH = Path("walmart_sales.csv")

## 2. Charger les données et paramètres du notebook source

On reprend le CSV du projet. On retire uniquement les lignes où la date ou la cible manquent/ne peuvent pas être interprétées, puis on applique la consigne des 3 sigmas aux quatre colonnes demandées : `temperature`, `fuel_price`, `cpi` et `unemployment`. Une ligne est considérée aberrante si au moins une de ces valeurs observées est hors de son intervalle $[\bar{x}-3\sigma,\bar{x}+3\sigma]$. Les valeurs manquantes de ces variables restent présentes pour être imputées dans le pipeline.

In [18]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Fichier introuvable : {DATA_PATH.resolve()}. Ouvrir le notebook depuis le dossier WALMART_SALES."
    )

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip().str.lower()

required_columns = {
    "store", "date", "weekly_sales", "holiday_flag",
    "temperature", "fuel_price", "cpi", "unemployment",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans le CSV : {sorted(missing_columns)}")

print(f"Données brutes : {df.shape[0]} lignes, {df.shape[1]} colonnes")
display(df.head())

Données brutes : 150 lignes, 8 colonnes


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092


In [19]:
# Conversion explicite : dates illisibles et cibles non numériques seront exclues.
df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y", errors="coerce")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

before_required = len(df)
df = df.dropna(subset=["date", TARGET]).copy()
print(f"Lignes conservées après contrôle Date/cible : {len(df)} / {before_required}")

# Consigne projet : supprimer les lignes dont au moins une variable mesurée
# parmi Temperature, Fuel_Price, CPI, Unemployment est à plus de 3 écarts-types.
outlier_features = ["temperature", "fuel_price", "cpi", "unemployment"]
for column in outlier_features:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Moyenne et écart-type calculés sur les valeurs observées (NaN ignorés).
means = df[outlier_features].mean()
stds = df[outlier_features].std(ddof=1)
lower_bounds = means - 3 * stds
upper_bounds = means + 3 * stds

# Valeurs manquantes ne déclenchent pas le masque d'outliers et seront imputées.
within_bounds = df[outlier_features].isna() | df[outlier_features].ge(lower_bounds) & df[outlier_features].le(upper_bounds)
keep_rows = within_bounds.all(axis=1)
removed_outliers = int((~keep_rows).sum())
df = df.loc[keep_rows].copy()
print(f"Lignes retirées par la règle des 3 sigma : {removed_outliers}")
print(f"Lignes conservées après filtre : {len(df)}")

# Reprise de la création des variables de date du notebook source.
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.dayofweek
df = df.drop(columns="date")

numeric_features = [
    "temperature", "fuel_price", "cpi", "unemployment",
    "year", "month", "day", "weekday",
]
categorical_features = ["store", "holiday_flag"]

print("Dimensions après préparation :", df.shape)
print("Valeurs manquantes utiles :")
display(df[numeric_features + categorical_features + [TARGET]].isna().sum())
print("Effectifs par magasin :")
display(df["store"].value_counts(dropna=False).sort_index())

Lignes conservées après contrôle Date/cible : 118 / 150
Lignes retirées par la règle des 3 sigma : 5
Lignes conservées après filtre : 113
Dimensions après préparation : (113, 11)
Valeurs manquantes utiles :


temperature     10
fuel_price      11
cpi              9
unemployment    11
year             0
month            0
day              0
weekday          0
store            0
holiday_flag     9
weekly_sales     0
dtype: int64

Effectifs par magasin :


store
1.0      8
2.0      6
3.0     10
4.0      6
5.0      7
6.0      6
7.0      7
8.0      6
9.0      4
10.0     3
11.0     1
13.0     9
14.0     9
15.0     3
16.0     4
17.0     5
18.0     7
19.0     8
20.0     4
Name: count, dtype: int64

## 3. Reproduire les problèmes rencontrés

Le problème observé était principalement une catégorie de magasin absente d’un pli d’entraînement : `OneHotEncoder()` avec son comportement par défaut lève alors une erreur. `GridSearchCV` reporte ensuite des scores `NaN`. L’ancienne sortie montrait aussi un `ConvergenceWarning` de Lasso, problème distinct. La cellule suivante repère les magasins potentiellement inconnus dans les plis, sans faire échouer volontairement le notebook.

In [20]:
X = df.drop(columns=TARGET)
y = df[TARGET]

# Stratifier le split externe par magasin seulement si chaque groupe peut figurer
# dans les deux partitions et si les deux partitions ont assez de places.
store_counts = X["store"].value_counts(dropna=False)
n_classes = len(store_counts)
n_test = int(np.ceil(TEST_SIZE * len(X)))
n_train = len(X) - n_test
can_stratify = (
    store_counts.min() >= 2
    and n_test >= n_classes
    and n_train >= n_classes
)
stratify_values = X["store"] if can_stratify else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_values,
)
print(f"Split train/test stratifié par magasin : {can_stratify}")
print(f"Train : {X_train.shape}; test : {X_test.shape}")

cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train), start=1):
    stores_fit = set(X_train.iloc[train_idx]["store"].dropna())
    stores_valid = set(X_train.iloc[valid_idx]["store"].dropna())
    unseen = sorted(stores_valid - stores_fit)
    print(f"Pli {fold} — magasins inconnus dans validation : {unseen}")

Split train/test stratifié par magasin : False
Train : (90, 10); test : (23, 10)
Pli 1 — magasins inconnus dans validation : [8.0]
Pli 2 — magasins inconnus dans validation : []
Pli 3 — magasins inconnus dans validation : []
Pli 4 — magasins inconnus dans validation : [10.0, 11.0]
Pli 5 — magasins inconnus dans validation : []


## 4. Diagnostiquer les causes avec des vérifications ciblées

Le split stratifié protège autant que possible la représentation de chaque magasin dans le train/test externe. La CV garde ses plis aléatoires reproductibles ; un magasin rare peut néanmoins manquer dans le train d’un pli. Cela est attendu sur un petit échantillon, et sera géré explicitement par l’encodeur. Les effectifs et la liste des catégories inconnues imprimés ci-dessus permettent de le constater.

In [21]:
print("Répartition magasins — train")
display(X_train["store"].value_counts(dropna=False).sort_index())
print("Répartition magasins — test")
display(X_test["store"].value_counts(dropna=False).sort_index())

if len(X_train) < N_SPLITS:
    raise ValueError(
        f"Il faut au moins {N_SPLITS} lignes d'entraînement pour une CV à {N_SPLITS} plis."
    )

print("Manquants numériques train :", int(X_train[numeric_features].isna().sum().sum()))
print("Manquants catégoriels train :", int(X_train[categorical_features].isna().sum().sum()))

Répartition magasins — train


store
1.0     6
2.0     5
3.0     9
4.0     5
5.0     7
6.0     4
7.0     6
8.0     1
9.0     4
10.0    1
11.0    1
13.0    8
14.0    8
15.0    2
16.0    3
17.0    4
18.0    6
19.0    6
20.0    4
Name: count, dtype: int64

Répartition magasins — test


store
1.0     2
2.0     1
3.0     1
4.0     1
6.0     2
7.0     1
8.0     5
10.0    2
13.0    1
14.0    1
15.0    1
16.0    1
17.0    1
18.0    1
19.0    2
Name: count, dtype: int64

Manquants numériques train : 34
Manquants catégoriels train : 7


## 5. Corriger le code en conservant sa logique

Corrections ciblées : le prétraitement reste dans le pipeline et est donc réajusté uniquement sur les données d’entraînement de chaque pli ; `handle_unknown='ignore'` évite l’échec lorsqu’un magasin manque dans un pli ; les variables numériques et la cible de Lasso sont standardisées pendant l’ajustement. Le target scaling est encapsulé dans `TransformedTargetRegressor`, donc il est appris à l’intérieur de chaque pli sans fuite de cible.

In [22]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [23]:
# Fixer le vocabulaire des catégories à partir de X complet (sans la cible)
# réserve les mêmes modalités dans chaque pli et élimine les warnings pour les magasins
# qui existent dans le CSV, même s'ils manquent dans un train de pli.
store_categories = np.sort(df["store"].dropna().unique())
holiday_categories = np.sort(df["holiday_flag"].dropna().unique())

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        categories=[store_categories, holiday_categories],
        drop="first",
        handle_unknown="ignore",
    )),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [24]:
# La cible Walmart est de l'ordre du million. Pour rendre l'optimisation Lasso
# mieux conditionnée, on standardise y dans chaque fit CV puis on inverse la
# transformation pour les prédictions et les métriques (qui restent en dollars).
lasso_estimator = TransformedTargetRegressor(
    regressor=Lasso(max_iter=100_000, tol=1e-3),
    transformer=StandardScaler(),
)
ridge_estimator = TransformedTargetRegressor(
    regressor=Ridge(),
    transformer=StandardScaler(),
)

pipelines = {
    "Linear Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]),
    "Ridge": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", ridge_estimator),
    ]),
    "Lasso": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", lasso_estimator),
    ]),
}

# Alpha est défini sur la cible standardisée pour Ridge/Lasso.
alpha_grid = np.logspace(-4, 1, 8)
param_grids = {
    "Linear Regression": {"model__fit_intercept": [True]},
    "Ridge": {"model__regressor__alpha": alpha_grid},
    "Lasso": {"model__regressor__alpha": alpha_grid},
}

scoring = {
    "R2": "r2",
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
}

## 6. Vérifier les résultats et les cas limites

Chaque modèle est optimisé uniquement sur `X_train` avec la même CV reproductible. Le jeu de test n’est utilisé qu’après le choix des hyperparamètres. `error_score='raise'` est volontaire : une nouvelle vraie erreur arrête la recherche au lieu d’être silencieusement convertie en score manquant.

In [25]:
grid_results = {}
summary_rows = []

for model_name, pipeline in pipelines.items():
    print(f"GridSearch : {model_name}")
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        scoring=scoring,
        refit="R2",
        cv=cv,
        n_jobs=-1,
        error_score="raise",
        return_train_score=False,
    )
    grid.fit(X_train, y_train)
    grid_results[model_name] = grid

    best_index = grid.best_index_
    y_test_pred = grid.best_estimator_.predict(X_test)
    summary_rows.append({
        "Model": model_name,
        "Best params": grid.best_params_,
        "CV R2 mean": grid.cv_results_["mean_test_R2"][best_index],
        "CV R2 std": grid.cv_results_["std_test_R2"][best_index],
        "Test R2": r2_score(y_test, y_test_pred),
        "Test MAE": mean_absolute_error(y_test, y_test_pred),
        "Test RMSE": mean_squared_error(y_test, y_test_pred) ** 0.5,
    })
    print("Meilleurs paramètres :", grid.best_params_)
    print(f"R² CV : {grid.best_score_:.4f}")

results = (
    pd.DataFrame(summary_rows)
    .sort_values("CV R2 mean", ascending=False)
    .reset_index(drop=True)
)
display(results)

GridSearch : Linear Regression
Meilleurs paramètres : {'model__fit_intercept': True}
R² CV : 0.9195
GridSearch : Ridge
Meilleurs paramètres : {'model__regressor__alpha': 0.07196856730011521}
R² CV : 0.9234
GridSearch : Lasso
Meilleurs paramètres : {'model__regressor__alpha': 0.0005179474679231213}
R² CV : 0.9233


,Model,Best params,CV R2 mean,CV R2 std,Test R2,Test MAE,Test RMSE
0,Ridge,{'model__regressor__alpha': 0.07196856730011521},0.923424,0.025302,0.924014,109262.797879,160610.912751
1,Lasso,{'model__regressor__alpha': 0.0005179474679231...,0.923264,0.025414,0.928603,109381.967967,155686.235980
2,Linear Regression,{'model__fit_intercept': True},0.919549,0.025142,0.931897,106980.738691,152052.004146


In [27]:
# Baseline demandé : évaluer la régression linéaire sur les ensembles train et test.
baseline = grid_results["Linear Regression"].best_estimator_
y_train_pred = baseline.predict(X_train)
y_test_pred = baseline.predict(X_test)

baseline_metrics = pd.DataFrame(
    {
        "Train": [
            r2_score(y_train, y_train_pred),
            mean_absolute_error(y_train, y_train_pred),
            mean_squared_error(y_train, y_train_pred) ** 0.5,
        ],
        "Test": [
            r2_score(y_test, y_test_pred),
            mean_absolute_error(y_test, y_test_pred),
            mean_squared_error(y_test, y_test_pred) ** 0.5,
        ],
    },
    index=["R²", "MAE", "RMSE"],
)
print("Performances du modèle baseline — Linear Regression")
display(baseline_metrics.round(4))

# Coefficients du modèle après prétraitement, associés aux noms des variables encodées.
feature_names = baseline.named_steps["preprocessor"].get_feature_names_out()
coefficients = pd.DataFrame(
    {
        "Feature": feature_names,
        "Coefficient": baseline.named_steps["model"].coef_,
    }
)
coefficients["Absolute coefficient"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values("Absolute coefficient", ascending=False)
print("Coefficients les plus influents du modèle baseline")
display(coefficients.head(20).round(4))
print("Intercept :", round(float(baseline.named_steps["model"].intercept_), 4))

Performances du modèle baseline — Linear Regression


,Train,Test
R²,0.9754,0.9319
MAE,80602.4187,106980.7387
RMSE,108522.2207,152052.0041


Coefficients les plus influents du modèle baseline


,Feature,Coefficient,Absolute coefficient
11,cat__store_5.0,-1.280409e+06,1.280409e+06
9,cat__store_3.0,-1.179209e+06,1.179209e+06
15,cat__store_9.0,-1.127185e+06,1.127185e+06
21,cat__store_16.0,-1.041555e+06,1.041555e+06
13,cat__store_7.0,-9.358714e+05,9.358714e+05
16,cat__store_10.0,6.776986e+05,6.776986e+05
10,cat__store_4.0,6.677310e+05,6.677310e+05
14,cat__store_8.0,-6.660744e+05,6.660744e+05
20,cat__store_15.0,-6.603363e+05,6.603363e+05
19,cat__store_14.0,6.444843e+05,6.444843e+05


Intercept : 1486353.3859


### Interprétation du baseline

Le score $R^2$ passe de 0,975 sur le train à 0,932 sur le test : l’écart indique un peu de surapprentissage, mais le score test reste élevé. Le baseline semble donc utile, sans conclure définitivement avec seulement 113 observations et un seul split. Les coefficients sont exprimés dans l’espace transformé : les variables numériques sont standardisées, et les modalités `store` sont des indicatrices comparées à la catégorie de référence supprimée par `drop="first"`. Leur amplitude ne doit donc pas être interprétée comme un effet par unité d’origine ni comme une importance causale.

### Comment lire les avertissements

- `Found unknown categories` : les plis CV peuvent ne pas contenir tous les magasins dans leur entraînement. `handle_unknown='ignore'` traite le cas sans erreur ; l’effet propre d’un magasin absent du pli ne peut toutefois pas être appris dans ce pli.
- `ConvergenceWarning` : c’est un sujet distinct des catégories. Ici, les variables numériques et la cible sont standardisées dans le pipeline et Lasso dispose de 100 000 itérations avec une tolérance de $10^{-3}$. Si un avertissement de convergence subsiste à l’exécution, il doit être lu et diagnostiqué, pas masqué ; vérifier le paramètre alpha sélectionné et l’avertissement exact.
- Aucun score CV ne devrait être `NaN`. Une exception arrête volontairement la cellule grâce à `error_score='raise'`.

La stratification du split externe stabilise la composition train/test par magasin lorsqu’elle est faisable. Elle ne contraint pas les plis internes de `KFold` ; les catégories inconnues restent donc un cas légitime à gérer.

**Précision pour ce notebook corrigé :** les catégories présentes dans le CSV complet sont déclarées explicitement avant le GridSearch. Les magasins connus qui manquent dans un pli ne génèrent donc plus le warning `Found unknown categories`. Un niveau réellement absent du CSV reste toléré par `handle_unknown='ignore'`, mais son effet ne peut être estimé.

## 7. Nettoyer et exécuter le notebook de bout en bout

Ordre d’exécution recommandé : **Restart Kernel**, puis **Run All**. Le notebook ne dépend pas de variables préexistantes du notebook source. Il applique le filtre des 3 sigma demandé, garde les magasins rares (plutôt que de supprimer arbitrairement des lignes) et retire les essais hors pipeline ainsi que les cellules de classification, qui ne correspondent pas à une cible de ventes continue. Le CSV est petit ; les scores sont exploratoires et sensibles au split. Pour une évaluation strictement sans fuite, les seuils des 3 sigma devraient être estimés uniquement sur les données d’entraînement ; ici ils sont calculés avant le split afin de reproduire directement la consigne de nettoyage du projet.

Vérification finale facultative après l’exécution :

In [26]:
assert len(X_train) + len(X_test) == len(df)
assert set(grid_results) == {"Linear Regression", "Ridge", "Lasso"}
assert results["CV R2 mean"].notna().all()
assert results["Test R2"].notna().all()
print("Vérifications terminées : split complet, 3 modèles ajustés, scores finis.")

Vérifications terminées : split complet, 3 modèles ajustés, scores finis.
